# LumenY 7 — HF Microstructure Features (5-min output)

Reuses **features_2** (already computed hourly microstructure features from notebooks_5).

Pipeline:
1. Load hourly features from `features_2/{PAIR}_microstructure.parquet`
2. Shift index +1 hour (leakage prevention: features from H:00-H:59 available at H+1:00)
3. Forward-fill to 5-min frequency
4. Filter to valid 5m bar timestamps
5. Add short-horizon labels (5m, 15m, 1h returns)

- **Pairs:** 7 majors only (tight spreads)
- **Features:** 64 microstructure (from features_2)
- **Output:** `features_7/` at 5-min frequency

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FEATURES_2_DIR = Path('../backend/data/features_2')
PROCESSED_DIR = Path('../backend/data/processed')
FEATURES_DIR = Path('../backend/data/features_7')
FEATURES_DIR.mkdir(exist_ok=True)

PAIRS = ['EURUSD', 'GBPUSD', 'AUDUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDJPY']

print(f'Input: {FEATURES_2_DIR}')
print(f'Output: {FEATURES_DIR}')
print(f'Pairs: {PAIRS}')

## 1. Load features_2, shift +1h, resample to 5min

In [ ]:
pair_shapes = {}

for pair in PAIRS:
    print(f'\n{"=" * 60}')
    print(f'Processing {pair}')
    print(f'{"=" * 60}')
    
    # ── Step 1: Load hourly features from features_2 ──
    feat_path = FEATURES_2_DIR / f'{pair}_microstructure.parquet'
    df_hourly = pd.read_parquet(feat_path)
    if 'datetime' in df_hourly.columns:
        df_hourly = df_hourly.set_index('datetime')
    df_hourly.index = pd.to_datetime(df_hourly.index)
    df_hourly = df_hourly.sort_index()
    
    # Drop 'pair' column if present (we'll re-add it later)
    if 'pair' in df_hourly.columns:
        df_hourly = df_hourly.drop(columns=['pair'])
    
    # Drop any label columns from features_2
    label_cols_to_drop = [c for c in df_hourly.columns if c.startswith('label_')]
    if label_cols_to_drop:
        df_hourly = df_hourly.drop(columns=label_cols_to_drop)
    
    print(f'  Hourly features: {df_hourly.shape}')
    print(f'  Range: {df_hourly.index.min()} to {df_hourly.index.max()}')
    
    # ── Step 2: Shift +1 hour (leakage prevention) ──
    # Features computed from bars H:00-H:59 are only available at H+1:00
    df_hourly.index = df_hourly.index + pd.Timedelta(hours=1)
    
    # ── Step 3: Resample to 5-min via forward-fill ──
    df_5min = df_hourly.resample('5min').ffill()
    
    # ── Step 4: Filter to valid 5m bar timestamps ──
    df_5m_bars = pd.read_parquet(PROCESSED_DIR / f'{pair}_5m.parquet')
    if 'datetime' in df_5m_bars.columns:
        df_5m_bars = df_5m_bars.set_index('datetime')
    df_5m_bars.index = pd.to_datetime(df_5m_bars.index)
    
    df_5min = df_5min[df_5min.index.isin(df_5m_bars.index)]
    
    # Add pair column
    df_5min['pair'] = pair
    
    # Convert to float32
    float_cols = df_5min.select_dtypes(include=[np.float64]).columns
    if len(float_cols) > 0:
        df_5min[float_cols] = df_5min[float_cols].astype(np.float32)
    
    # Save per-pair
    out_path = FEATURES_DIR / f'{pair}_microstructure.parquet'
    df_5min.to_parquet(out_path)
    
    pair_shapes[pair] = df_5min.shape
    print(f'  5-min features: {df_5min.shape}')
    print(f'  Range: {df_5min.index.min()} to {df_5min.index.max()}')
    print(f'  Memory: {df_5min.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    print(f'  Saved: {out_path}')
    
    del df_hourly, df_5min, df_5m_bars

print(f'\n{"=" * 60}')
print('ALL PAIRS COMPLETE')
print(f'{"=" * 60}')
for pair, shape in pair_shapes.items():
    print(f'  {pair}: {shape[0]:,} rows x {shape[1]} features')

## 2. Combine & Add Labels

In [ ]:
# Combine all pairs
dfs = []
for pair in PAIRS:
    path = FEATURES_DIR / f'{pair}_microstructure.parquet'
    df = pd.read_parquet(path)
    dfs.append(df)
    print(f'{pair}: {len(df):,} rows')

df_all = pd.concat(dfs).sort_index()
del dfs
print(f'\nCombined: {df_all.shape}')

# ── Add 5-min labels ──
label_rows = []
for pair in PAIRS:
    df_5m = pd.read_parquet(PROCESSED_DIR / f'{pair}_5m.parquet')
    if 'datetime' in df_5m.columns:
        df_5m = df_5m.set_index('datetime')
    df_5m.index = pd.to_datetime(df_5m.index)
    
    # Next 5-min return (1 bar forward)
    df_5m['label_5m'] = np.log(df_5m['close'].shift(-1) / df_5m['close'])
    # Next 15-min return (3 bars forward)
    df_5m['label_15m'] = np.log(df_5m['close'].shift(-3) / df_5m['close'])
    # Next 1-hour return (12 bars forward)
    df_5m['label_1h'] = np.log(df_5m['close'].shift(-12) / df_5m['close'])
    
    df_5m['pair'] = pair
    label_rows.append(df_5m[['pair', 'label_5m', 'label_15m', 'label_1h']])

df_labels = pd.concat(label_rows).sort_index()
del label_rows

# Merge labels into features
df_all = df_all.reset_index()
df_labels = df_labels.reset_index()
df_labels.columns = ['datetime', 'pair_label', 'label_5m', 'label_15m', 'label_1h']

df_all = df_all.merge(
    df_labels,
    left_on=['datetime', 'pair'],
    right_on=['datetime', 'pair_label'],
    how='left'
).drop(columns=['pair_label'])

df_all = df_all.set_index('datetime').sort_index()

for col in ['label_5m', 'label_15m', 'label_1h']:
    df_all[col] = df_all[col].astype(np.float32)

print(f'\nFinal dataset: {df_all.shape}')
print(f'Date range: {df_all.index.min().date()} to {df_all.index.max().date()}')
print(f'Memory: {df_all.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Labels available: {df_all["label_5m"].notna().sum():,} rows')

feature_cols = [c for c in df_all.columns if c not in ['pair', 'label_5m', 'label_15m', 'label_1h']]
print(f'\nFeature columns ({len(feature_cols)}):')
for c in sorted(feature_cols):
    print(f'  {c}')

In [ ]:
# Save combined dataset
out_path = FEATURES_DIR / 'all_pairs_microstructure.parquet'
df_all.to_parquet(out_path)
file_size = out_path.stat().st_size / 1e6
print(f'Saved: {out_path}')
print(f'File size: {file_size:.1f} MB')

## 3. Feature Quality Check

In [ ]:
feature_cols = [c for c in df_all.columns if c not in ['pair', 'label_5m', 'label_15m', 'label_1h']]

print('Feature Statistics')
print('=' * 80)

nan_pct = df_all[feature_cols].isna().mean() * 100
print(f'\n{"Feature":<35} {"NaN%":>8} {"Mean":>12} {"Std":>12} {"Min":>12} {"Max":>12}')
print('-' * 95)
for col in sorted(feature_cols):
    vals = df_all[col].dropna()
    if len(vals) > 0:
        print(f'{col:<35} {nan_pct[col]:>7.1f}% {vals.mean():>12.6f} {vals.std():>12.6f} {vals.min():>12.6f} {vals.max():>12.6f}')
    else:
        print(f'{col:<35} {100.0:>7.1f}% {"N/A":>12} {"N/A":>12} {"N/A":>12} {"N/A":>12}')

# Correlation with short-horizon labels
print(f'\n\nCorrelation with label_5m (top features):')
print('-' * 50)
df_valid = df_all.dropna(subset=['label_5m'])
corrs = {}
for col in feature_cols:
    valid = df_valid[[col, 'label_5m']].dropna()
    if len(valid) > 100:
        corrs[col] = valid[col].corr(valid['label_5m'])

sorted_corrs = sorted(corrs.items(), key=lambda x: abs(x[1]), reverse=True)
for col, corr in sorted_corrs[:20]:
    print(f'  {col:<35} r = {corr:>8.5f}')

In [ ]:
import matplotlib.pyplot as plt

key_features = ['hurst_6h', 'kyle_lambda', 'entropy_norm', 'jump_ratio',
                'vr_5', 'order_imbalance', 'noise_to_signal', 'vol_of_vol',
                'realized_skew', 'epps_1m_5m', 'ret_concentration', 'amihud_illiq']

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.patch.set_facecolor('#080c14')

for ax, feat in zip(axes.flat, key_features):
    ax.set_facecolor('#080c14')
    if feat in df_all.columns:
        vals = df_all[feat].dropna()
        lo, hi = vals.quantile(0.01), vals.quantile(0.99)
        vals_clipped = vals.clip(lo, hi)
        ax.hist(vals_clipped, bins=50, color='#4fc3f7', alpha=0.7, edgecolor='none')
    ax.set_title(feat, color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

plt.suptitle('HF Microstructure Feature Distributions (5-min output)', color='white', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

- **Source:** `features_2/` (pre-computed hourly microstructure from notebooks_5)
- **Leakage prevention:** Features shifted +1 hour (H:00-H:59 bars → available at H+1:00)
- **Output:** Forward-filled to 5-min, filtered to valid 5m bar timestamps
- **Labels:** `label_5m`, `label_15m`, `label_1h` (true 5-min resolution)
- **Runtime:** ~30 seconds (no feature computation, just load + resample)